# 05 Model Training

Train baselines and machine learning models for future 7-day xwOBA.



In [19]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


In [20]:
model_path = PROJECT_ROOT / "data/features/model_ready_2025_04_01_to_05_31.csv"

df = pd.read_csv(model_path)
df["prediction_date"] = pd.to_datetime(df["prediction_date"])

print(df.shape)
df.head()

(18350, 23)


,batter,prediction_date,last_7d_PA,last_7d_xwOBA,last_7d_wOBA,last_7d_avg_exit_velocity,last_7d_avg_launch_angle,last_14d_PA,last_14d_xwOBA,last_14d_wOBA,...,season_xwOBA,season_wOBA,season_avg_exit_velocity,season_avg_launch_angle,future_7d_PA,future_7d_xwOBA,xwOBA_diff_7d_vs_season,wOBA_diff_7d_vs_season,xwOBA_diff_14d_vs_season,wOBA_diff_14d_vs_season
0,518595,2025-04-02,5,0.243464,0.18,92.080000,23.400000,5,0.243464,0.18,...,0.243464,0.18,92.080000,23.400000,6,0.263810,0.0,0.0,0.0,0.0
1,545361,2025-04-02,5,0.427303,0.33,89.500000,27.333333,5,0.427303,0.33,...,0.427303,0.33,89.500000,27.333333,24,0.420977,0.0,0.0,0.0,0.0
2,571448,2025-04-02,5,0.103000,0.36,95.075000,-1.250000,5,0.103000,0.36,...,0.103000,0.36,95.075000,-1.250000,22,0.241077,0.0,0.0,0.0,0.0
3,621043,2025-04-02,5,0.103800,0.00,97.033333,3.000000,5,0.103800,0.00,...,0.103800,0.00,97.033333,3.000000,25,0.304311,0.0,0.0,0.0,0.0
4,621439,2025-04-02,5,0.343233,0.32,92.500000,35.000000,5,0.343233,0.32,...,0.343233,0.32,92.500000,35.000000,21,0.187231,0.0,0.0,0.0,0.0


In [21]:
df.columns.tolist()

['batter',
 'prediction_date',
 'last_7d_PA',
 'last_7d_xwOBA',
 'last_7d_wOBA',
 'last_7d_avg_exit_velocity',
 'last_7d_avg_launch_angle',
 'last_14d_PA',
 'last_14d_xwOBA',
 'last_14d_wOBA',
 'last_14d_avg_exit_velocity',
 'last_14d_avg_launch_angle',
 'season_PA',
 'season_xwOBA',
 'season_wOBA',
 'season_avg_exit_velocity',
 'season_avg_launch_angle',
 'future_7d_PA',
 'future_7d_xwOBA',
 'xwOBA_diff_7d_vs_season',
 'wOBA_diff_7d_vs_season',
 'xwOBA_diff_14d_vs_season',
 'wOBA_diff_14d_vs_season']

In [22]:
df["future_7d_xwOBA"].describe()

count    18350.000000
mean         0.313295
std          0.099286
min          0.000000
25%          0.248339
50%          0.309385
75%          0.373892
max          0.874359
Name: future_7d_xwOBA, dtype: float64

In [23]:
print(df["prediction_date"].min())
print(df["prediction_date"].max())

2025-04-02 00:00:00
2025-05-30 00:00:00


In [24]:
target_col = "future_7d_xwOBA"

feature_cols = [
    "last_7d_PA",
    "last_7d_xwOBA",
    "last_7d_wOBA",
    "last_7d_avg_exit_velocity",
    "last_7d_avg_launch_angle",

    "last_14d_PA",
    "last_14d_xwOBA",
    "last_14d_wOBA",
    "last_14d_avg_exit_velocity",
    "last_14d_avg_launch_angle",

    "season_PA",
    "season_xwOBA",
    "season_wOBA",
    "season_avg_exit_velocity",
    "season_avg_launch_angle",

    "xwOBA_diff_7d_vs_season",
    "wOBA_diff_7d_vs_season",
    "xwOBA_diff_14d_vs_season",
    "wOBA_diff_14d_vs_season",
]

missing_features = [c for c in feature_cols if c not in df.columns]
print("Missing features:", missing_features)

Missing features: []


In [25]:
split_date = pd.Timestamp("2025-05-11")

train_df = df[df["prediction_date"] < split_date].copy()
test_df = df[df["prediction_date"] >= split_date].copy()

print("Train:", train_df.shape, train_df["prediction_date"].min(), train_df["prediction_date"].max())
print("Test:", test_df.shape, test_df["prediction_date"].min(), test_df["prediction_date"].max())

Train: (12225, 23) 2025-04-02 00:00:00 2025-05-10 00:00:00
Test: (6125, 23) 2025-05-11 00:00:00 2025-05-30 00:00:00


In [26]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(X_train.shape, X_test.shape)

(12225, 19) (6125, 19)


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Spearman": spearmanr(y_true, y_pred, nan_policy="omit").correlation,
    }

In [28]:
baseline_results = {}

for baseline_col in ["season_xwOBA", "last_7d_xwOBA", "last_14d_xwOBA"]:
    y_pred = test_df[baseline_col]
    baseline_results[baseline_col] = regression_metrics(y_test, y_pred)

baseline_results_df = pd.DataFrame(baseline_results).T
baseline_results_df

,MAE,RMSE,R2,Spearman
season_xwOBA,0.085315,0.109931,-0.041941,0.184931
last_7d_xwOBA,0.105830,0.134957,-0.570330,0.116480
last_14d_xwOBA,0.093395,0.120114,-0.243911,0.152088


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]
)

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

ridge_metrics = regression_metrics(y_test, ridge_pred)
ridge_metrics

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/henry_tsai/Desktop/d

{'MAE': 0.08231161242934144,
 'RMSE': np.float64(0.10640817388577145),
 'R2': 0.023776545426812068,
 'Spearman': np.float64(0.18474494734311112)}

In [30]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=5,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_metrics = regression_metrics(y_test, rf_pred)
rf_metrics

{'MAE': 0.08283212985683466,
 'RMSE': np.float64(0.10718845767710435),
 'R2': 0.009406893696948915,
 'Spearman': np.float64(0.17879275772716058)}

In [31]:
all_results = baseline_results.copy()
all_results["Ridge Regression"] = ridge_metrics
all_results["Random Forest"] = rf_metrics

results_df = pd.DataFrame(all_results).T
results_df.sort_values("RMSE")

,MAE,RMSE,R2,Spearman
Ridge Regression,0.082312,0.106408,0.023777,0.184745
Random Forest,0.082832,0.107188,0.009407,0.178793
season_xwOBA,0.085315,0.109931,-0.041941,0.184931
last_14d_xwOBA,0.093395,0.120114,-0.243911,0.152088
last_7d_xwOBA,0.105830,0.134957,-0.570330,0.116480


In [32]:
pred_df = test_df[["batter", "prediction_date", target_col]].copy()

pred_df["pred_season_xwOBA"] = test_df["season_xwOBA"].values
pred_df["pred_last_7d_xwOBA"] = test_df["last_7d_xwOBA"].values
pred_df["pred_last_14d_xwOBA"] = test_df["last_14d_xwOBA"].values
pred_df["pred_ridge"] = ridge_pred
pred_df["pred_random_forest"] = rf_pred

pred_df.head()

,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest
12225,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462
12226,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146
12227,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416
12228,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217
12229,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948


In [33]:
results_df

,MAE,RMSE,R2,Spearman
season_xwOBA,0.085315,0.109931,-0.041941,0.184931
last_7d_xwOBA,0.105830,0.134957,-0.570330,0.116480
last_14d_xwOBA,0.093395,0.120114,-0.243911,0.152088
Ridge Regression,0.082312,0.106408,0.023777,0.184745
Random Forest,0.082832,0.107188,0.009407,0.178793


In [34]:
from src.data_loader import save_csv

pred_path = PROJECT_ROOT / "data/predictions/model_predictions_2025_04_01_to_05_31.csv"
save_csv(pred_df, pred_path)

metrics_path = PROJECT_ROOT / "reports/model_metrics_2025_04_01_to_05_31.csv"
save_csv(results_df.reset_index().rename(columns={"index": "model"}), metrics_path)

print(pred_path.exists())
print(metrics_path.exists())

True
True


In [35]:
def top_k_summary(pred_df, pred_col, actual_col="future_7d_xwOBA", ks=[10, 20, 50]):
    rows = []
    for k in ks:
        top = pred_df.sort_values(pred_col, ascending=False).head(k)
        rows.append({
            "model": pred_col,
            "k": k,
            "actual_mean_xwOBA": top[actual_col].mean(),
            "actual_median_xwOBA": top[actual_col].median(),
            "overall_mean_xwOBA": pred_df[actual_col].mean(),
        })
    return pd.DataFrame(rows)

In [36]:
topk_results = pd.concat([
    top_k_summary(pred_df, "pred_season_xwOBA"),
    top_k_summary(pred_df, "pred_last_7d_xwOBA"),
    top_k_summary(pred_df, "pred_last_14d_xwOBA"),
    top_k_summary(pred_df, "pred_ridge"),
    top_k_summary(pred_df, "pred_random_forest"),
])

topk_results

,model,k,actual_mean_xwOBA,actual_median_xwOBA,overall_mean_xwOBA
0,pred_season_xwOBA,10,0.352201,0.366729,0.313882
1,pred_season_xwOBA,20,0.338698,0.358452,0.313882
2,pred_season_xwOBA,50,0.361930,0.361482,0.313882
0,pred_last_7d_xwOBA,10,0.325600,0.309834,0.313882
1,pred_last_7d_xwOBA,20,0.349061,0.352865,0.313882
2,pred_last_7d_xwOBA,50,0.338292,0.334357,0.313882
0,pred_last_14d_xwOBA,10,0.371846,0.388161,0.313882
1,pred_last_14d_xwOBA,20,0.380162,0.374443,0.313882
2,pred_last_14d_xwOBA,50,0.342464,0.337197,0.313882
0,pred_ridge,10,0.432312,0.411492,0.313882


In [38]:
from lightgbm import LGBMRegressor

lgbm_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            LGBMRegressor(
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
            ),
        ),
    ]
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)

lgbm_metrics = regression_metrics(y_test, lgbm_pred)
lgbm_metrics

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4333
[LightGBM] [Info] Number of data points in the train set: 12225, number of used features: 19
[LightGBM] [Info] Start training from score 0.313000


/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'MAE': 0.08384873588141067,
 'RMSE': np.float64(0.10867198121443009),
 'R2': -0.018203128406499847,
 'Spearman': np.float64(0.14644157852667752)}

In [39]:
all_results["LightGBM"] = lgbm_metrics

results_df = pd.DataFrame(all_results).T
results_df.sort_values("RMSE")

,MAE,RMSE,R2,Spearman
Ridge Regression,0.082312,0.106408,0.023777,0.184745
Random Forest,0.082832,0.107188,0.009407,0.178793
LightGBM,0.083849,0.108672,-0.018203,0.146442
season_xwOBA,0.085315,0.109931,-0.041941,0.184931
last_14d_xwOBA,0.093395,0.120114,-0.243911,0.152088
last_7d_xwOBA,0.105830,0.134957,-0.570330,0.116480


In [40]:
pred_df["pred_lightgbm"] = lgbm_pred
pred_df.head()

,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
12225,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462,0.276183
12226,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146,0.347680
12227,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416,0.327371
12228,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217,0.322703
12229,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948,0.282033


In [41]:
def top_k_summary(pred_df, pred_col, actual_col="future_7d_xwOBA", ks=[10, 20, 50, 100]):
    rows = []

    overall_mean = pred_df[actual_col].mean()
    overall_median = pred_df[actual_col].median()

    for k in ks:
        top = pred_df.sort_values(pred_col, ascending=False).head(k)

        rows.append({
            "model": pred_col,
            "k": k,
            "top_k_actual_mean_xwOBA": top[actual_col].mean(),
            "top_k_actual_median_xwOBA": top[actual_col].median(),
            "overall_mean_xwOBA": overall_mean,
            "overall_median_xwOBA": overall_median,
            "lift_vs_overall_mean": top[actual_col].mean() - overall_mean,
        })

    return pd.DataFrame(rows)

In [42]:
topk_results = pd.concat([
    top_k_summary(pred_df, "pred_season_xwOBA"),
    top_k_summary(pred_df, "pred_last_7d_xwOBA"),
    top_k_summary(pred_df, "pred_last_14d_xwOBA"),
    top_k_summary(pred_df, "pred_ridge"),
    top_k_summary(pred_df, "pred_random_forest"),
    top_k_summary(pred_df, "pred_lightgbm"),
], ignore_index=True)

topk_results

,model,k,top_k_actual_mean_xwOBA,top_k_actual_median_xwOBA,overall_mean_xwOBA,overall_median_xwOBA,lift_vs_overall_mean
0,pred_season_xwOBA,10,0.352201,0.366729,0.313882,0.30946,0.038319
1,pred_season_xwOBA,20,0.338698,0.358452,0.313882,0.30946,0.024816
2,pred_season_xwOBA,50,0.361930,0.361482,0.313882,0.30946,0.048048
3,pred_season_xwOBA,100,0.352087,0.345952,0.313882,0.30946,0.038205
4,pred_last_7d_xwOBA,10,0.325600,0.309834,0.313882,0.30946,0.011718
5,pred_last_7d_xwOBA,20,0.349061,0.352865,0.313882,0.30946,0.035179
6,pred_last_7d_xwOBA,50,0.338292,0.334357,0.313882,0.30946,0.024410
7,pred_last_7d_xwOBA,100,0.333457,0.327841,0.313882,0.30946,0.019575
8,pred_last_14d_xwOBA,10,0.371846,0.388161,0.313882,0.30946,0.057963
9,pred_last_14d_xwOBA,20,0.380162,0.374443,0.313882,0.30946,0.066280


In [43]:
topk_pivot = topk_results.pivot(
    index="model",
    columns="k",
    values="top_k_actual_mean_xwOBA"
)

topk_pivot

k,10,20,50,100
model,,,,
pred_last_14d_xwOBA,0.371846,0.380162,0.342464,0.333528
pred_last_7d_xwOBA,0.325600,0.349061,0.338292,0.333457
pred_lightgbm,0.370876,0.368048,0.348230,0.348699
pred_random_forest,0.388643,0.381876,0.362765,0.355797
pred_ridge,0.432312,0.380747,0.368248,0.378426
pred_season_xwOBA,0.352201,0.338698,0.361930,0.352087


In [44]:
topk_lift_pivot = topk_results.pivot(
    index="model",
    columns="k",
    values="lift_vs_overall_mean"
)

topk_lift_pivot

k,10,20,50,100
model,,,,
pred_last_14d_xwOBA,0.057963,0.066280,0.028581,0.019645
pred_last_7d_xwOBA,0.011718,0.035179,0.024410,0.019575
pred_lightgbm,0.056993,0.054165,0.034348,0.034816
pred_random_forest,0.074760,0.067993,0.048883,0.041915
pred_ridge,0.118429,0.066864,0.054366,0.064544
pred_season_xwOBA,0.038319,0.024816,0.048048,0.038205


In [45]:
ridge_coef = ridge_model.named_steps["model"].coef_

ridge_coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": ridge_coef
}).sort_values("coefficient", ascending=False)

ridge_coef_df

,feature,coefficient
9,last_14d_avg_launch_angle,0.011685
0,last_7d_PA,0.010190
13,season_avg_exit_velocity,0.008581
10,season_PA,0.005325
11,season_xwOBA,0.004008
6,last_14d_xwOBA,0.003323
8,last_14d_avg_exit_velocity,0.002866
1,last_7d_xwOBA,0.002770
16,wOBA_diff_7d_vs_season,0.001568
3,last_7d_avg_exit_velocity,0.000630


In [46]:
ridge_coef_df["abs_coefficient"] = ridge_coef_df["coefficient"].abs()

ridge_coef_df.sort_values("abs_coefficient", ascending=False)

,feature,coefficient,abs_coefficient
9,last_14d_avg_launch_angle,0.011685,0.011685
0,last_7d_PA,0.010190,0.010190
13,season_avg_exit_velocity,0.008581,0.008581
14,season_avg_launch_angle,-0.008578,0.008578
5,last_14d_PA,-0.007142,0.007142
10,season_PA,0.005325,0.005325
11,season_xwOBA,0.004008,0.004008
4,last_7d_avg_launch_angle,-0.003789,0.003789
6,last_14d_xwOBA,0.003323,0.003323
8,last_14d_avg_exit_velocity,0.002866,0.002866


In [47]:
lgbm_feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": lgbm_model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

lgbm_feature_importance

,feature,importance
14,season_avg_launch_angle,1251
13,season_avg_exit_velocity,1131
12,season_wOBA,988
11,season_xwOBA,950
9,last_14d_avg_launch_angle,942
10,season_PA,891
3,last_7d_avg_exit_velocity,876
8,last_14d_avg_exit_velocity,857
4,last_7d_avg_launch_angle,830
5,last_14d_PA,721


In [48]:
from src.data_loader import save_csv

metrics_path = PROJECT_ROOT / "reports/model_metrics_2025_04_01_to_05_31.csv"
save_csv(results_df.reset_index().rename(columns={"index": "model"}), metrics_path)

topk_path = PROJECT_ROOT / "reports/topk_results_2025_04_01_to_05_31.csv"
save_csv(topk_results, topk_path)

ridge_coef_path = PROJECT_ROOT / "reports/ridge_coefficients_2025_04_01_to_05_31.csv"
save_csv(ridge_coef_df, ridge_coef_path)

lgbm_importance_path = PROJECT_ROOT / "reports/lgbm_feature_importance_2025_04_01_to_05_31.csv"
save_csv(lgbm_feature_importance, lgbm_importance_path)

print(metrics_path.exists())
print(topk_path.exists())
print(ridge_coef_path.exists())
print(lgbm_importance_path.exists())

True
True
True
True


In [49]:
pred_df

,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
12225,455117,2025-05-11,0.148583,0.314537,0.440423,0.388523,0.312607,0.292462,0.276183
12226,456781,2025-05-11,0.419369,0.206825,0.204182,0.224426,0.294077,0.293146,0.347680
12227,457705,2025-05-11,0.274772,0.366160,0.399297,0.413562,0.327504,0.322416,0.327371
12228,457759,2025-05-11,0.443777,0.286520,0.261328,0.282639,0.301951,0.308217,0.322703
12229,467793,2025-05-11,0.322851,0.288303,0.337477,0.322596,0.322525,0.284948,0.282033
...,...,...,...,...,...,...,...,...,...
18345,695734,2025-05-30,0.646709,0.267324,0.267324,0.267324,0.320207,0.351382,0.341247
18346,700932,2025-05-30,0.307458,0.298016,0.242260,0.247151,0.345984,0.311371,0.275169
18347,702332,2025-05-30,0.169200,0.287145,0.338127,0.289082,0.327951,0.274107,0.259033
18348,805779,2025-05-30,0.234800,0.331056,0.307836,0.352186,0.333505,0.299058,0.318091


In [50]:
pred_path = PROJECT_ROOT / "data/predictions/model_predictions_2025_04_01_to_05_31.csv"
save_csv(pred_df, pred_path)

print(pred_path.exists())

True
